# 02 — Quantizer, Gaussian surrogate gradient: tests + PTQ baselines
Needs a Kaggle Dataset containing `qtta_lib.py` and `resnet20_fp32.pt`, plus the CIFAR-10 Python dataset.

In [ ]:
import sys, os, copy, math, glob
import torch, torch.nn as nn
sys.path.insert(0, os.path.dirname(glob.glob("/kaggle/input/**/qtta_lib.py", recursive=True)[0]))
import qtta_lib as Q
from qtta_lib import QCFG, FakeQuant, gauss_grad
dev = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)

## Unit tests (all must pass)

In [ ]:
qmin, qmax = -7, 7
scale = torch.tensor(0.5, device=dev)

# 1) forward is bit-identical to the plain quantizer for every backward mode
x = torch.randn(20000, device=dev) * 3
ref = torch.clamp(torch.round(x / scale), qmin, qmax) * scale
for mode, sig in [("ste", 0.5), ("gauss", 0.1), ("gauss", 1.0), ("gauss", 2.0)]:
    assert torch.equal(FakeQuant.apply(x, scale, qmin, qmax, mode, sig), ref), (mode, sig)
print("1) forward identical: OK")

# 2) truncated closed form == autograd of the exact smoothed quantizer  E[Q(u+eps)] = qmin + sum_t Phi((u-t)/sig)
def exact_smoothed(u, sig):
    t = torch.arange(qmin, qmax, device=u.device, dtype=u.dtype) + 0.5
    return qmin + torch.special.ndtr((u[..., None] - t) / sig).sum(-1)
u = torch.linspace(qmin - 3, qmax + 3, 4001, device=dev, dtype=torch.float64, requires_grad=True)
for sig in [0.1, 0.25, 0.5, 1.0, 2.0]:
    g_ref, = torch.autograd.grad(exact_smoothed(u, sig).sum(), u)
    g_imp = gauss_grad(u.detach(), qmin, qmax, sig)
    err = (g_ref - g_imp).abs().max().item()
    assert err < 5e-3, (sig, err)
    print(f"2) sigma/Delta={sig:<5} max|exact-impl| = {err:.1e}")

# 3) wide smoothing -> identity gradient away from the clip edges
ui = torch.linspace(-2, 2, 1001, device=dev)
assert (gauss_grad(ui, qmin, qmax, 1.0) - 1).abs().max() < 1e-2
print("3) large sigma -> ~identity: OK")

# 4) saturated region has zero gradient in both modes
uo = torch.tensor([-12., -9., 9., 12.], device=dev)
assert gauss_grad(uo, qmin, qmax, 0.5).abs().max() < 1e-6
print("4) zero gradient when saturated: OK")

# 5) gradient sparsity vs sigma/Delta (fraction of near-zero entries, u uniform in range)
uu = torch.empty(200000, device=dev).uniform_(qmin, qmax)
for sig in [0.1, 0.25, 0.5, 1.0, 2.0]:
    g = gauss_grad(uu, qmin, qmax, sig)
    print(f"5) sigma/Delta={sig:<5} mean g={g.mean():.3f}  frac(g<0.05)={(g < 0.05).float().mean():.3f}")

## Load FP32 model + data, PTQ baselines

In [ ]:
fp, ck = Q.load_fp32(Q.find_file("resnet20_fp32.pt"), dev)
train_loader, test_loader = Q.get_cifar10(bs=500)
print("FP32 acc:", Q.evaluate(fp, test_loader, dev), "| saved:", ck["final_test_acc"])

qmodels = {}
for name, (wb, ab) in {"W8A8": (8, 8), "W4A8": (4, 8), "W4A4": (4, 4)}.items():
    qm, ql = Q.quantize_model(fp, wb, ab, train_loader, dev)
    qmodels[name] = qm
    print(f"{name}: clean acc = {Q.evaluate(qm, test_loader, dev):.2f}")

## Sanity: BN-parameter entropy gradients, STE vs Gaussian (W4A4, one clean batch)

In [ ]:
def bn_params(m): return [p for mod in m.modules() if isinstance(mod, nn.BatchNorm2d) for p in (mod.weight, mod.bias)]
def entropy(lg): return -(lg.softmax(1) * lg.log_softmax(1)).sum(1).mean()

m = copy.deepcopy(qmodels["W4A4"]).train()          # BN uses batch stats (TENT-style)
for p in m.parameters(): p.requires_grad_(False)
for p in bn_params(m): p.requires_grad_(True)
xb, _ = next(iter(test_loader)); xb = xb[:128].to(dev)

def gvec(mode, sig):
    QCFG.mode, QCFG.sigma = mode, sig
    m.zero_grad(set_to_none=True); entropy(m(xb)).backward()
    return torch.cat([p.grad.flatten() for p in bn_params(m)]).clone()

g_ste = gvec("ste", 0)
print(f"STE        |g|={g_ste.norm():.4f}")
for sig in [0.1, 0.25, 0.5, 1.0, 2.0]:
    g = gvec("gauss", sig)
    cos = torch.nn.functional.cosine_similarity(g, g_ste, dim=0).item()
    print(f"gauss {sig:<5} |g|={g.norm():.4f}  cos(g, g_ste)={cos:.3f}")
QCFG.mode = "ste"